In [9]:

# --------------------------------------------------------------
# Imports and configuration
# --------------------------------------------------------------
import os
import numpy as np
import pandas as pd

# Base directory for the 10 CSVs. Edit to match your environment.
BASE_DIR = "/Users/chrismader/Python/SLDS/Out"

# File name pattern. Files are expected to exist as:
#   {BASE_DIR}/gridsearch_results1.csv
#   ...
#   {BASE_DIR}/gridsearch_results10.csv
FILENAME_PATTERN = "gridsearch_results{}.csv"

# Value to aggregate
VALUE_COL = "cagr_rel_ex_ante"

# Grouping keys (match prior tables)
IDX = ["config", "n_regimes", "dim_latent"]

# How to identify "unrestricted" vs "restricted" configs (heuristic used before)
def is_unrestricted(cfg):
    s = str(cfg)
    return s.startswith("[")  # unrestricted examples looked like "[y,g,v]" etc.


In [10]:

# --------------------------------------------------------------
# Load 10 files
# --------------------------------------------------------------
dfs = {}
missing = []
for i in range(1, 11):
    path = os.path.join(BASE_DIR, FILENAME_PATTERN.format(i))
    if os.path.exists(path):
        df = pd.read_csv(path)
        df["__file_id__"] = i
        dfs[i] = df
    else:
        missing.append(path)

if missing:
    print("Warning: missing files:")
    for p in missing:
        print("  -", p)

assert len(dfs) > 0, "No input files found. Set BASE_DIR correctly."


In [19]:

# --------------------------------------------------------------
# Aggregation helpers
# --------------------------------------------------------------
def per_file_means(dfs, mask=None):
    """For each file i, compute mean(VALUE_COL) across ALL securities
    within each (config, n_regimes, dim_latent). Optionally filter with mask.
    Returns a dict: i -> DataFrame with columns IDX + [f"avg{i}"]."""
    out = {}
    for i, df in dfs.items():
        dfi = df
        if mask is not None:
            dfi = dfi.loc[mask(dfi)]
        for c in IDX + [VALUE_COL]:
            if c not in dfi.columns:
                raise KeyError(f"Column '{c}' missing in file {i}.")
        g = (dfi.groupby(IDX, dropna=False)[VALUE_COL]
                .mean()
                .rename(f"avg{i}")
                .to_frame()
                .reset_index())
        out[i] = g
    return out

def combine_10_averages(avg_dict):
    """Join avg1..avg10 on IDX. Then compute summary columns across these 10 columns:
    avg, std, min, max, range. Sort by avg desc."""
    keys = sorted(avg_dict.keys())
    base = avg_dict[keys[0]].copy()
    for k in keys[1:]:
        base = base.merge(avg_dict[k], on=IDX, how="outer")
    avg_cols = [f"avg{i}" for i in keys]
    base[avg_cols] = base[avg_cols].apply(pd.to_numeric, errors="coerce")
    base["avg"]   = base[avg_cols].mean(axis=1, skipna=True)
    base["std"]   = base[avg_cols].std(axis=1, ddof=1, skipna=True)
    base["min"]   = base[avg_cols].min(axis=1, skipna=True)
    base["max"]   = base[avg_cols].max(axis=1, skipna=True)
    base["range"] = base["max"] - base["min"]
    base = base[IDX + avg_cols + ["avg","std","min","max","range"]]
    base = base.sort_values("avg", ascending=False).reset_index(drop=True)
    base = base.round(4)
    return base

def save_csv(df, name, out_dir=None):
    if out_dir is None:
        out_dir = BASE_DIR
    path = os.path.join(out_dir, f"{name}.csv")
    df.to_csv(path, index=False)
    print("Saved:", path)


In [20]:

# --------------------------------------------------------------
# Masks for subsets
# --------------------------------------------------------------
def mask_unrestricted(df):
    return df["config"].apply(is_unrestricted)

def mask_restricted(df):
    return ~df["config"].apply(is_unrestricted)

def mask_all(df):
    return pd.Series(True, index=df.index)


In [24]:

# --------------------------------------------------------------
# Compute tables
# --------------------------------------------------------------
# Unrestricted
avg_u = per_file_means(dfs, mask=mask_unrestricted)
tbl_u = combine_10_averages(avg_u)
save_csv(tbl_u, "pivot_unrestricted_10x")
# tbl_u  # display



Saved: /Users/chrismader/Python/SLDS/Out/pivot_unrestricted_10x.csv


In [25]:

# Restricted
avg_r = per_file_means(dfs, mask=mask_restricted)
tbl_r = combine_10_averages(avg_r)
save_csv(tbl_r, "pivot_restricted_10x")
# tbl_r  # display


Saved: /Users/chrismader/Python/SLDS/Out/pivot_restricted_10x.csv


In [26]:

# All configs
avg_all = per_file_means(dfs, mask=mask_all)
tbl_all = combine_10_averages(avg_all)
save_csv(tbl_all, "pivot_all_10x")
tbl_all  # display


Saved: /Users/chrismader/Python/SLDS/Out/pivot_all_10x.csv


,config,n_regimes,dim_latent,avg1,avg2,avg3,avg4,avg5,avg6,avg7,avg8,avg9,avg10,avg,std,min,max,range
0,[y],6,1,0.0604,0.0627,0.066,0.064,0.0717,0.0694,0.0712,0.0576,0.0567,0.0704,0.065,0.0056,0.0567,0.0717,0.015


In [ ]:
# boostrap: subsample of securities 80pct 76